# Server interaction with AsyncSSH


In [1]:
import asyncssh
from pathlib import Path

## Using SSH to execute remote commands

**Important note:** some servers have a limit on the number of successful or unsuccessful logins for a time period.  If that limit is exceeded, your IP address will be banned for some time.  Hence be careful never to make a large number of connections over a short period of time!

**Important note:** this notebook disables host key verification by passing `known_hosts=None` to AsyncSSH so the example stays compact.  In production code, you should specify a file that contains the known hosts, e.g., `~/.ssh/known_hosts`.

Create a client connection and connect to the system.


In [2]:
username = 'vsc30140'

In [3]:
hostname = 'login.hpc.kuleuven.be'

In [4]:
conn = await asyncssh.connect(
    hostname,
    username=username,
    known_hosts=None,
)


Execute an `ls` command on the remote host.

In [5]:
result = await conn.run('ls -l *.slurm', check=False)

Now the output of the remote `ls` command can be read from the result object.  In this case we ignore the exit status and standard error.


In [6]:
print(result.stdout.rstrip())


-rw-r----- 1 vsc30140 vsc30140 468 Jun 17  2025 ext4_conda.slurm
-rw-r----- 1 vsc30140 vsc30140 518 Jun 17  2025 ext4_ssd_conda.slurm
-rw-r----- 1 vsc30140 vsc30140 226 Nov 21  2024 heterogenous_job.slurm
-rw-r----- 1 vsc30140 vsc30140 756 May 26  2025 install_r_packages.slurm
-rw-r----- 1 vsc30140 vsc30140 151 Oct 13 15:01 jobscript.slurm
-rw-r----- 1 vsc30140 vsc30140 315 Jun 27  2024 modulus_build.slurm
-rw-r----- 1 vsc30140 vsc30140 327 Jun 17  2025 vsc_data_conda.slurm


### Dealing with errors

You can check the exit status of the command that was executed, the next one will not exit succesfully.  You can also access standard error.

In [7]:
result = await conn.run(
    '/bin/ls -l this_file_certainly_does_not_exists.txt 2>&1',
    check=False,
    request_pty=False,
)

In [8]:
result.exit_status

2

In [9]:
print(repr(result.stdout.rstrip()))
print(repr(result.stderr.rstrip()))

"ls: cannot access 'this_file_certainly_does_not_exists.txt': No such file or directory"
''


### Using standard input

Do an `wc` on a local file that will be standard input to the remote command, reading the result back in.

In [10]:
process = await conn.create_process('wc')

Send the input to the remote `wc` by writing to `stdin`.  Once done, the stream should be closed by sending EOF.


In [11]:
with open('wc.py', 'r', encoding='utf-8') as file:
    for line in file:
        process.stdin.write(line)
process.stdin.write_eof()

Now the output from the command can be read from `stdout`.

In [12]:
stdout, stderr = await process.communicate()
print(stdout.rstrip())
print(stderr.rstrip())

     71     170    1693



Compare to the local result.

In [13]:
!wc wc.py

  71  170 1693 wc.py


In [14]:
conn.close()
await conn.wait_closed()

## Using SFTP to transfer files

In [15]:
conn = await asyncssh.connect(
    hostname,
    username=username,
    known_hosts=None,
)

Open an SFTP client to the remote host, and put a file on the system.

In [16]:
sftp_client = await conn.start_sftp_client()

Define paths to the input file and output file.

In [17]:
input_path = Path('wc.py')
output_path = Path('wc_py_wc.txt')

In [18]:
await sftp_client.put(str(input_path), str(input_path))

Check whether the input file is on the remote host.

In [19]:
result = await conn.run(f'ls -l {input_path}', check=False)
print(result.stdout.rstrip())
print(result.stderr.rstrip())

-rw-r--r-- 1 vsc30140 vsc30140 1693 Mar 25 15:53 wc.py



Compute the word count, redirecting the output to a file.

In [20]:
result = await conn.run(
    f'wc {input_path} > {output_path}',
    check=False,
)


Show standard error.

In [21]:
print(result.stderr.rstrip())


Transfer the result back to the local host.

In [22]:
await sftp_client.get(str(output_path), str(output_path))


In [23]:
!cat wc_py_wc.txt

  71  170 1693 wc.py


Remove the local output file.

In [24]:
output_path.unlink()

Remove the remote input and output files.

In [25]:
_ = await conn.run(f'rm {input_path} {output_path}', check=False)

Close the SFTP client and the SSH client.

In [26]:
sftp_client.exit()
conn.close()
await conn.wait_closed()